# Ensemble ("Mixed-Use") Building Types

Real buildings are often not well represented by a single BuildStock building type. A building that is
70% office space over 30% ground-floor retail, or an apartment building with street-level shops, doesn't
match any single ComStock/ResStock prototype -- but it can be approximated as a **weighted combination**
of the building types that *are* available.

`EnsembleBuildingType` models exactly that: a synthetic "building" made up of two or more
`(product, building_type)` components, each contributing a `fraction` of the combined energy profile
(fractions must sum to 1.0). This notebook shows:

1. Defining an ensemble and validating/normalizing its fractions.
2. Pulling real time series for each component and auto-stitching them into one combined ensemble time
   series with `pull_ensemble_time_series()`.
3. A same-product (ComStock-only) example: 70% MediumOffice / 30% RetailStripmall.
4. A cross-product example mixing ComStock (commercial) and ResStock (residential) components: ground-floor
   retail under apartments -- exactly the kind of mixed-use property that the ENERGY STAR crosswalk's
   "Mixed Use Property" entry can only approximate with a single best-fit building type.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from buildstock_processor import (
    EnsembleBuildingType,
    combine_ensemble_time_series,
    map_energy_star_property_type,
    normalize_time_series_columns,
    pull_ensemble_time_series,
)

dataset_path = Path().resolve().cwd() / "datasets"
ensemble_dir = dataset_path / "ensemble"
ensemble_dir.mkdir(parents=True, exist_ok=True)
print(f"Ensemble dataset path: {ensemble_dir}")

## 1. Defining an ensemble

An ensemble needs at least 2 components, and every component's `fraction` must be in `(0, 1]`.

In [ ]:
office_retail = EnsembleBuildingType.from_fractions(
    "70% MediumOffice / 30% RetailStripmall",
    {
        ("comstock", "MediumOffice"): 0.7,
        ("comstock", "RetailStripmall"): 0.3,
    },
)
for component in office_retail.components:
    print(f"{component.product:10s} {component.building_type:20s} {component.fraction:.0%}")
print("total fraction:", office_retail.total_fraction)

Fractions don't have to sum to exactly 1.0 at construction time -- that's only enforced when you actually combine time series (`assert_normalized()`/`combine_ensemble_time_series()`). If your shares were entered as percentages that don't sum perfectly due to rounding, fix them up with `.normalized()`:

In [ ]:
rounded_thirds = EnsembleBuildingType.from_fractions(
    "Rounded thirds",
    {
        ("comstock", "SmallOffice"): 0.333,
        ("comstock", "MediumOffice"): 0.333,
        ("comstock", "RetailStripmall"): 0.333,
    },
)
print("before normalizing:", rounded_thirds.total_fraction)
print("after normalizing: ", rounded_thirds.normalized().total_fraction)

## 2. Pulling and auto-stitching real time series

`pull_ensemble_time_series()` downloads one representative building's time series per component (via `BuildStockProcessor.process_building_time_series()`, same as any other BuildStock time series download) and combines them into a single synthetic ensemble time series, weighted by each component's `fraction`. It returns both the combined result and each component's own raw series, so you can compare the blend against its ingredients.

In [ ]:
energy_columns = [
    "out.electricity.total.energy_consumption",
    "out.natural_gas.total.energy_consumption",
    "out.site_energy.total.energy_consumption",
]

combined, components = pull_ensemble_time_series(
    office_retail,
    save_dir=ensemble_dir,
    state="DE",
    value_columns=energy_columns,
)

print("Combined ensemble time series:")
display(combined.head())

for key, series in components.items():
    print(key, "->", series.shape)

### Compare a sample winter day: components vs. combined ensemble

In [ ]:
fig, axes = plt.subplots(len(energy_columns), 1, figsize=(11, 8), sharex=True)

day_start, day_end = "2018-01-15", "2018-01-16"
combined_day = combined.set_index("timestamp").loc[day_start:day_end]

for ax, column in zip(axes, energy_columns):
    for key, series in components.items():
        product, building_type = key
        series_day = normalize_time_series_columns(series).set_index("timestamp").loc[day_start:day_end]
        label = f"{building_type} ({product})"
        ax.plot(series_day.index, series_day[column], label=label, alpha=0.6, linestyle="--")
    ax.plot(combined_day.index, combined_day[column], label="Combined ensemble", color="black", linewidth=2)
    ax.set_ylabel(column.split(".")[1])
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Timestamp")
fig.suptitle(f'"{office_retail.name}" -- Jan 15 load shapes')
fig.tight_layout()
plt.show()

### Annual totals: components vs. combined ensemble

In [ ]:
annual_totals = {"Combined ensemble": combined[energy_columns].sum()}
for key, series in components.items():
    product, building_type = key
    annual_totals[f"{building_type} ({product})"] = normalize_time_series_columns(series)[energy_columns].sum()

annual_totals_df = pd.DataFrame(annual_totals).T
display(annual_totals_df)

annual_totals_df.plot(kind="bar", figsize=(9, 5), title=f'"{office_retail.name}" -- annual energy by source')
plt.ylabel("Annual energy consumption (kWh)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 3. Mixing ComStock and ResStock: ground-floor retail under apartments

Ensembles aren't limited to a single product. A very common real building type -- street-level retail with apartments above -- can be modeled by mixing a ComStock retail component with a ResStock multifamily component. ComStock and ResStock publish the same 15-minute time series layout for the currently supported releases, but ResStock's energy columns carry a `..kwh`-style unit suffix that ComStock's don't (e.g. `out.electricity.total.energy_consumption..kwh` vs. `out.electricity.total.energy_consumption`); `combine_ensemble_time_series()` normalizes both to the same bare column name automatically, so this works exactly like the same-product example above.

In [ ]:
retail_over_apartments = EnsembleBuildingType.from_fractions(
    "55% Ground-Floor Retail / 45% Apartments",
    {
        ("comstock", "RetailStripmall"): 0.55,
        ("resstock", "Multi-Family with 5+ Units"): 0.45,
    },
)

mixed_combined, mixed_components = pull_ensemble_time_series(
    retail_over_apartments,
    save_dir=ensemble_dir,
    state="DE",
    value_columns=energy_columns,
)

mixed_annual_totals = {"Combined ensemble": mixed_combined[energy_columns].sum()}
for key, series in mixed_components.items():
    product, building_type = key
    mixed_annual_totals[f"{building_type} ({product})"] = normalize_time_series_columns(series)[energy_columns].sum()

mixed_annual_totals_df = pd.DataFrame(mixed_annual_totals).T
display(mixed_annual_totals_df)

mixed_annual_totals_df.plot(kind="bar", figsize=(9, 5), title=f'"{retail_over_apartments.name}" -- annual energy by source')
plt.ylabel("Annual energy consumption (kWh)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

### Why this matters: ENERGY STAR's "Mixed Use Property" only has a single best-fit BuildStock match

The packaged ENERGY STAR crosswalk (`map_energy_star_property_type()`) maps "Mixed Use Property" to a single approximate BuildStock type, since the crosswalk (by design) picks one best-fit type per ENERGY STAR property type. An `EnsembleBuildingType` lets you go further for a specific known property: model its *actual* use mix (e.g. retail + residential, at whatever split you know it to be) instead of falling back to one generic anchor use.

In [ ]:
mapping = map_energy_star_property_type("Mixed Use Property")
print(mapping)

## Combining already-downloaded time series directly

If you've already downloaded component time series yourself (e.g. via `BuildStockProcessor.process_building_time_series()` for specific buildings you picked), you can skip the download step and call `combine_ensemble_time_series()` directly:

In [ ]:
# `components` (returned above by pull_ensemble_time_series) is already a
# {(product, building_type): DataFrame} mapping -- exactly what combine_ensemble_time_series() expects.
recombined = combine_ensemble_time_series(office_retail, components, value_columns=energy_columns)
recombined.equals(combined)